In [1]:
import pandas as pd
import glob
import os

print("--- SENTRi-X: CIC-IDS2017 ETL Pipeline ---")
print("Step 1: Loading the Enterprise Network Data\n")

# 1. Point to your CIC-IDS2017 raw folder
data_dir = "../data/raw/cic_ids2017/"

# 2. Use glob to find all the daily CSV files (Monday, Tuesday, etc.)
all_files = glob.glob(os.path.join(data_dir, "*.csv"))

if not all_files:
    print("ERROR: No CSV files found! Double-check the folder path.")
else:
    print(f"Successfully located {len(all_files)} CSV chunks.")
    
    df_list = []
    for file in all_files:
        filename = os.path.basename(file)
        print(f"Extracting {filename}...")
        # cp1252 encoding handles the weird hidden characters in this specific dataset
        chunk = pd.read_csv(file, encoding='cp1252', low_memory=False)
        df_list.append(chunk)

    # 3. Stitch the days together into one massive matrix
    df_cic = pd.concat(df_list, ignore_index=True)
    
    # 4. The "Sanity" Clean: Strip hidden whitespace from all column names
    df_cic.columns = df_cic.columns.str.strip()

    print("\n--- Extraction Complete! ---")
    print(f"Total Enterprise Network Flows: {df_cic.shape[0]:,}")
    print(f"Total Features: {df_cic.shape[1]}")
    
    # 5. Let's see what kind of attacks we are dealing with
    print("\nTarget Variable Distribution ('Label'):")
    print(df_cic['Label'].value_counts())

--- SENTRi-X: CIC-IDS2017 ETL Pipeline ---
Step 1: Loading the Enterprise Network Data

Successfully located 8 CSV chunks.
Extracting Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
Extracting Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
Extracting Friday-WorkingHours-Morning.pcap_ISCX.csv...
Extracting Monday-WorkingHours.pcap_ISCX.csv...
Extracting Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
Extracting Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
Extracting Tuesday-WorkingHours.pcap_ISCX.csv...
Extracting Wednesday-workingHours.pcap_ISCX.csv...

--- Extraction Complete! ---
Total Enterprise Network Flows: 2,830,743
Total Features: 79

Target Variable Distribution ('Label'):
Label
BENIGN                          2273097
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator           

In [2]:
import numpy as np

print("--- Step 2: Data Cleaning and Target Standardization ---")

# 1. Handle CICFlowMeter's notorious Infinity and NaN values
print("Purging infinite values and NaNs generated by CICFlowMeter...")
# Replace infinities with NaN, then drop all NaNs
df_cic.replace([np.inf, -np.inf], np.nan, inplace=True)

initial_rows = len(df_cic)
df_cic.dropna(inplace=True)
final_rows = len(df_cic)
print(f"Dropped {initial_rows - final_rows:,} corrupted rows.")
print(f"Cleaned Dataset Size: {final_rows:,} flows.")

# 2. Standardize the Target Variable
# SENTRi-X needs a binary 'label' column (0 for Normal, 1 for Attack)
print("\nMapping Multi-class labels to Binary Target ('label')...")
df_cic['label'] = df_cic['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# Drop the old text label to prevent data leakage
df_cic.drop(columns=['Label'], inplace=True)

print("\nNew Threat Distribution ('label'):")
print(df_cic['label'].value_counts())
print(f"\nTarget mapping complete. Columns remaining: {df_cic.shape[1]}")

--- Step 2: Data Cleaning and Target Standardization ---
Purging infinite values and NaNs generated by CICFlowMeter...
Dropped 2,867 corrupted rows.
Cleaned Dataset Size: 2,827,876 flows.

Mapping Multi-class labels to Binary Target ('label')...

New Threat Distribution ('label'):
label
0    2271320
1     556556
Name: count, dtype: int64

Target mapping complete. Columns remaining: 79


In [7]:
print("--- SENTRi-X Diagnostic: Unmasking the Columns ---")
# Print the first 20 columns wrapped in quotes
for i, col in enumerate(df_cic.columns[:20]):
    print(f"{i}: '{col}'")

--- SENTRi-X Diagnostic: Unmasking the Columns ---
0: 'destination port'
1: 'flow duration'
2: 'total fwd packets'
3: 'total backward packets'
4: 'total length of fwd packets'
5: 'total length of bwd packets'
6: 'fwd packet length max'
7: 'fwd packet length min'
8: 'fwd packet length mean'
9: 'fwd packet length std'
10: 'bwd packet length max'
11: 'bwd packet length min'
12: 'bwd packet length mean'
13: 'bwd packet length std'
14: 'flow bytes/s'
15: 'flow packets/s'
16: 'flow iat mean'
17: 'flow iat std'
18: 'flow iat max'
19: 'flow iat min'


In [ ]:
print("--- Step 3: The Universal Schema Mapper ---")

# 1. Force lowercase to standardize
df_cic.columns = df_cic.columns.str.lower()

# 2. Hardcoded renaming based exactly on your diagnostic output
schema_mapping = {
    'total length of fwd packets': 'src_bytes',
    'total length of bwd packets': 'dst_bytes',
    'total fwd packets': 'src_pkts',
    'total backward packets': 'dst_pkts',
    'flow duration': 'duration'
}
df_cic_translated = df_cic.rename(columns=schema_mapping)

# 3. Inject missing "Ghost" Columns
print("Injecting missing ToN-IoT features (including 'proto') as baseline values...")
missing_zeros = ['src_ip_bytes', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode']
for col in missing_zeros:
    df_cic_translated[col] = 0

# Since your CSVs are missing the protocol column entirely, we inject a dummy state
df_cic_translated['proto'] = 'other'

# Inject a neutral connection state since CIC uses raw TCP flags instead
df_cic_translated['conn_state'] = 'OTH'

# 4. Filter down to ONLY the baseline columns our AI knows how to read
final_columns = [
    'src_bytes', 'dst_bytes', 'src_pkts', 'dst_pkts', 'duration',
    'src_ip_bytes', 'dst_ip_bytes', 'conn_state', 'proto',
    'dns_qclass', 'dns_qtype', 'dns_rcode', 'label'
]

df_cic_final = df_cic_translated[final_columns]

print("\n--- Translation Complete! ---")
print("New Standardized Schema Ready for the Pipeline:")
print(df_cic_final.dtypes)

--- Step 3: The Universal Schema Mapper (Hardcoded Edition) ---
Injecting missing ToN-IoT features (including 'proto') as baseline values...

--- Translation Complete! ---
New Standardized Schema Ready for the Pipeline:
src_bytes       int64
dst_bytes       int64
src_pkts        int64
dst_pkts        int64
duration        int64
src_ip_bytes    int64
dst_ip_bytes    int64
conn_state        str
proto             str
dns_qclass      int64
dns_qtype       int64
dns_rcode       int64
label           int64
dtype: object
